# Solar Panel Defect Classifier: Training Notebook (PyTorch, Python 3.14)

**Task:** classify a solar panel photo into one of 6 conditions using transfer learning (MobileNetV2).

**Dataset:** *Solar panel clean and faulty images* (Kaggle, `pythonafroz/solar-panel-clean-and-faulty-images`)
Classes: `Bird-drop`, `Clean`, `Dusty`, `Electrical-damage`, `Physical-Damage`, `Snow-Covered`

**Pipeline**
1. Download the dataset (kagglehub) and locate the class folders
2. Explore the data (class counts, sample images)
3. Clean the file list and make a stratified 70/15/15 split
4. Preprocess (resize to 224x224, normalise) and augment
5. Train MobileNetV2: Phase 1 (frozen backbone), Phase 2 (fine-tuning)
6. Evaluate on the test set (accuracy, classification report, confusion matrix)
7. Save the model, class names and metrics into `model/` for the Streamlit app

Run the cells from top to bottom. After the last cell, start the app with `streamlit run app.py`.

## 1. Imports and configuration

In [ ]:
import os, sys, json, copy, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, os.getcwd())          # make model_utils.py importable
import model_utils as mu

warnings.filterwarnings("ignore")
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
DEVICE = mu.get_device()
print("Device :", DEVICE)

In [ ]:
# ----------------------------- CONFIGURATION -----------------------------
SEED = 42
BATCH_SIZE = 16
NUM_WORKERS = 0           # keep 0 on Windows/notebooks (avoids multiprocessing problems)

EPOCHS_PHASE1 = 12        # backbone frozen, train the new head
EPOCHS_PHASE2 = 10        # fine-tune the last blocks
LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-4
FINE_TUNE_FROM_BLOCK = 14 # MobileNetV2 has 19 feature blocks (0..18); unfreeze blocks 14..18
PATIENCE = 4              # early stopping patience (epochs)
PRETRAINED = True         # use ImageNet weights (downloaded once, about 14 MB)

KAGGLE_DATASET = "pythonafroz/solar-panel-clean-and-faulty-images"

# If you downloaded the dataset manually (or kagglehub fails), put the folder that
# contains the class folders here, e.g. r"C:\Users\you\Downloads\Faulty_solar_panel"
# Leave as None to use kagglehub automatically.
MANUAL_DATA_DIR = None

MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Get the dataset

By default the notebook downloads the dataset with `kagglehub` (about 330 MB, cached after the first run).
If that fails, download it manually from Kaggle, extract it, and set `MANUAL_DATA_DIR` in the cell above.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def count_images(folder: Path) -> int:
    return sum(1 for f in folder.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS)

def find_class_roots(base: str):
    """Find every folder whose sub-folders directly contain images (= class folders)."""
    candidates = []
    for dirpath, dirnames, _ in os.walk(base):
        subs = [Path(dirpath) / d for d in dirnames]
        class_subs = [s for s in subs if count_images(s) > 0]
        if len(class_subs) >= 2:
            candidates.append((Path(dirpath), class_subs))
    return candidates

if MANUAL_DATA_DIR:
    base_path = MANUAL_DATA_DIR
else:
    import kagglehub
    base_path = kagglehub.dataset_download(KAGGLE_DATASET)
print("Dataset location:", base_path)

candidates = find_class_roots(base_path)
if not candidates:
    raise RuntimeError("No class folders with images found. Check the dataset path.")

print("\nFolders that look like class roots:")
for root, subs in candidates:
    total = sum(count_images(s) for s in subs)
    print(f"  {root}  ->  {len(subs)} classes, {total} images")

In [ ]:
# Pick the class root. Priority: exactly 6 classes; if several candidates exist
# (for example an original and an augmented copy), take the one with the fewest images
# so that no image appears twice. You can override this by setting DATA_DIR yourself.
DATA_DIR = None   # e.g. Path(r"C:\path\to\Faulty_solar_panel")

if DATA_DIR is None:
    six = [c for c in candidates if len(c[1]) == 6] or candidates
    DATA_DIR = min(six, key=lambda c: sum(count_images(s) for s in c[1]))[0]

DATA_DIR = Path(DATA_DIR)
print("Using class root:", DATA_DIR)

## 3. Explore the data

In [ ]:
class_names = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir() and count_images(d) > 0])
num_classes = len(class_names)
print("Classes:", class_names)

# Build a table of (file path, label) and drop unreadable / corrupt files
rows, skipped = [], []
for idx, cname in enumerate(class_names):
    for f in sorted((DATA_DIR / cname).iterdir()):
        if f.suffix.lower() not in IMG_EXTS:
            continue
        try:
            with Image.open(f) as im:
                im.verify()
            rows.append((str(f), idx, cname))
        except Exception:
            skipped.append(str(f))

df = pd.DataFrame(rows, columns=["path", "label", "class"])
print(f"\nTotal usable images: {len(df)}   |   skipped (corrupt/unreadable): {len(skipped)}")
display(df["class"].value_counts().rename("images").to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, y="class", order=class_names, ax=ax, palette="viridis")
ax.set_title("Images per class")
plt.tight_layout()
plt.savefig(MODEL_DIR / "class_distribution.png", dpi=120)
plt.show()

In [ ]:
# A few random samples from each class
fig, axes = plt.subplots(num_classes, 4, figsize=(12, 3 * num_classes))
for r, cname in enumerate(class_names):
    subset = df[df["class"] == cname]
    sample = subset.sample(4, random_state=SEED, replace=len(subset) < 4)
    for c, p in enumerate(sample["path"]):
        axes[r, c].imshow(Image.open(p).convert("RGB"))
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(cname, loc="left", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(MODEL_DIR / "sample_images.png", dpi=100)
plt.show()

## 4. Stratified train / validation / test split (70 / 15 / 15)

Stratified means every class appears in the same proportion in all three splits.
The test set is never used during training. It is only used once, at the end.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print(f"Train: {len(train_df)}   Validation: {len(val_df)}   Test: {len(test_df)}")
display(pd.DataFrame({
    "train": train_df["class"].value_counts(),
    "val": val_df["class"].value_counts(),
    "test": test_df["class"].value_counts(),
}))

## 5. Preprocessing and data pipeline

* Every image is decoded as RGB and **resized to 224x224** (the MobileNetV2 input size), then normalised with the ImageNet mean and standard deviation.
* **Augmentation** (random horizontal flip, small rotation and zoom, slight contrast change) is applied to the **training set only**.
* Brightness changes are avoided on purpose, because strong brightness shifts could blur the difference between *Clean* and *Dusty*.
* The preprocessing code lives in `model_utils.py`, so the Streamlit app uses exactly the same steps.
* Class weights compensate for the slight class imbalance.

In [ ]:
class SolarDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

pin = DEVICE.type == "cuda"
train_loader = DataLoader(SolarDataset(train_df, mu.train_transform), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(SolarDataset(val_df, mu.eval_transform), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(SolarDataset(test_df, mu.eval_transform), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=train_df["label"].values)
class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print("Class weights:", {class_names[i]: round(float(w), 2) for i, w in enumerate(weights)})

## 6. Build the model (MobileNetV2 transfer learning)

MobileNetV2 is pre-trained on ImageNet and is small and fast, which suits a small dataset and a laptop CPU.
We replace its final layer with a new head for the 6 solar-panel classes.

In [ ]:
model = mu.build_model(num_classes, pretrained=PRETRAINED).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

total_params = sum(p.numel() for p in model.parameters())
print(f"MobileNetV2 with {num_classes}-class head | total parameters: {total_params:,}")
print(model.classifier)

In [ ]:
def set_train_mode(model, frozen_blocks):
    """Train mode for the model, but keep BatchNorm of the frozen blocks in eval mode."""
    model.train()
    for block in model.features[:frozen_blocks]:
        block.eval()

def run_epoch(model, loader, optimizer=None, frozen_blocks=0):
    training = optimizer is not None
    if training:
        set_train_mode(model, frozen_blocks)
    else:
        model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

def fit(model, optimizer, epochs, frozen_blocks, label):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
    history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
    best_loss, best_state, wait = float("inf"), copy.deepcopy(model.state_dict()), 0
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, frozen_blocks)
        va_loss, va_acc = run_epoch(model, val_loader)
        scheduler.step(va_loss)
        for k, v in zip(history, (tr_loss, va_loss, tr_acc, va_acc)):
            history[k].append(v)
        print(f"[{label}] epoch {epoch:2d}/{epochs}  loss {tr_loss:.4f}  acc {tr_acc:.3f}  |  "
              f"val_loss {va_loss:.4f}  val_acc {va_acc:.3f}  ({time.time() - t0:.0f}s)")
        if va_loss < best_loss - 1e-4:
            best_loss, best_state, wait = va_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"[{label}] early stopping")
                break
    model.load_state_dict(best_state)      # go back to the best epoch
    return history

## 7. Phase 1: train the new head (backbone frozen)

In [ ]:
for p in model.features.parameters():
    p.requires_grad = False

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE1)
history1 = fit(model, optimizer, EPOCHS_PHASE1, frozen_blocks=len(model.features), label="phase 1")

## 8. Phase 2: fine-tuning

We unfreeze the last blocks of MobileNetV2 (from block `FINE_TUNE_FROM_BLOCK`) and continue training with a **smaller learning rate**
so the pre-trained features are only adjusted slightly. BatchNorm layers of the frozen blocks stay in evaluation mode for stable training.

In [ ]:
for block in model.features[FINE_TUNE_FROM_BLOCK:]:
    for p in block.parameters():
        p.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE2)
history2 = fit(model, optimizer, EPOCHS_PHASE2, frozen_blocks=FINE_TUNE_FROM_BLOCK, label="phase 2")

In [ ]:
# Combine both phases into one set of training curves
def merge(key):
    return history1[key] + history2[key]

split_at = len(history1["accuracy"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (a, b, t) in zip(axes, [(merge("accuracy"), merge("val_accuracy"), "Accuracy"),
                                (merge("loss"), merge("val_loss"), "Loss")]):
    ax.plot(a, label="train"); ax.plot(b, label="validation")
    ax.axvline(split_at - 1, color="gray", linestyle="--", label="start fine-tuning")
    ax.set_title(t); ax.set_xlabel("Epoch"); ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / "training_curves.png", dpi=120)
plt.show()

## 9. Evaluate on the unseen test set

The test set was never used for training or model selection, so these numbers show how the model behaves on new images.

In [ ]:
model.eval()
y_true, y_prob = [], []
with torch.no_grad():
    for x, y in test_loader:
        probs = torch.softmax(model(x.to(DEVICE)), dim=1).cpu().numpy()
        y_prob.append(probs); y_true.append(y.numpy())
y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)
y_pred = y_prob.argmax(axis=1)

test_acc = float((y_pred == y_true).mean())
test_loss = float(-np.log(np.clip(y_prob[np.arange(len(y_true)), y_true], 1e-9, 1)).mean())
print(f"Test accuracy: {test_acc:.4f}   |   Test loss: {test_loss:.4f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Confusion matrix (counts)"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Greens", xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("Confusion matrix (row-normalised)"); axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(MODEL_DIR / "confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
# Look at some wrong predictions to understand the model's weak spots
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} wrong predictions out of {len(y_true)}")
if len(wrong):
    show = wrong[:8]
    cols = 4
    rows_n = int(np.ceil(len(show) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(14, 3.6 * rows_n))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for ax, i in zip(axes, show):
        ax.imshow(Image.open(test_df.iloc[i]["path"]).convert("RGB"))
        ax.set_title(f"True: {class_names[y_true[i]]}\nPred: {class_names[y_pred[i]]} ({y_prob[i].max():.0%})", fontsize=9)
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "wrong_predictions.png", dpi=100)
    plt.show()

## 10. Save everything the Streamlit app needs

In [ ]:
torch.save(model.state_dict(), MODEL_DIR / "solar_model.pt")

with open(MODEL_DIR / "class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)

report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
metrics = {
    "test_accuracy": test_acc,
    "test_loss": test_loss,
    "num_images": int(len(df)),
    "train_images": int(len(train_df)),
    "val_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "image_size": list(mu.IMG_SIZE),
    "backbone": "MobileNetV2 (ImageNet pre-trained, PyTorch)",
    "per_class": {c: {k: float(report_dict[c][k]) for k in ("precision", "recall", "f1-score")} for c in class_names},
}
with open(MODEL_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved files in", MODEL_DIR.resolve())
for p in sorted(MODEL_DIR.iterdir()):
    print("  ", p.name)

In [ ]:
# Sanity check: reload the saved model and predict one test image exactly the way the app does
reloaded = mu.load_trained_model(MODEL_DIR / "solar_model.pt", num_classes, DEVICE)
sample_path = test_df.iloc[0]["path"]
img = np.asarray(Image.open(sample_path).convert("RGB"), dtype=np.uint8)
probs = mu.predict_probs(reloaded, img, DEVICE)
print("File     :", sample_path)
print("True     :", test_df.iloc[0]["class"])
print("Predicted:", class_names[int(np.argmax(probs))], f"({probs.max():.1%})")

## Done

The `model/` folder now contains `solar_model.pt`, `class_names.json`, `metrics.json` and the evaluation plots.
Start the web app from a terminal in the project folder:

```
streamlit run app.py
```